# Portion-Aware LLM Pipeline — EDA & Comparison

Compares baseline v2 (`ingredient_match_llm.py`) vs portion-aware v3 (`ingredient_match_llm_portion.py`) on **100 recipes, seed=42**.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
BASELINE_DIR = ROOT / 'scratch' / 'recipe_matching_llm_100_baseline_rerun'
PORTION_DIR = ROOT / 'scratch' / 'recipe_matching_llm_100_portion'
FP_CSV = ROOT / 'Data' / 'All_Food_Data_April_2026' / 'food_portion.csv'

## 1. food_portion EDA — count labels & catalog overlap

In [ ]:
fp = pd.read_csv(FP_CSV)
print('rows:', len(fp), 'fdc_ids:', fp['fdc_id'].nunique())
print('measure_unit_id top:')
display(fp['measure_unit_id'].value_counts().head(8))

count_like = fp[(fp['measure_unit_id'] == 9999) & (fp['gram_weight'] > 0)]
print(f'count-like (9999): {len(count_like):,} rows, {count_like["fdc_id"].nunique():,} fdc_ids')

# Readable modifiers (exclude pure FNDDS numeric codes)
code_only = count_like['modifier'].astype(str).str.fullmatch(r'\d+')
readable = count_like[~code_only.fillna(True)]
top_mods = readable['modifier'].str.lower().value_counts().head(20)
print('Top count modifiers:')
display(top_mods)

fig, ax = plt.subplots(figsize=(8, 4))
top_mods.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top food_portion count modifiers (9999, non-numeric)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 2. Load experiment outputs

In [ ]:
baseline = pd.read_csv(BASELINE_DIR / 'ingredient_matches_llm.csv')
portion = pd.read_csv(PORTION_DIR / 'ingredient_matches_llm.csv')
comparison = pd.read_csv(PORTION_DIR / 'comparison_merged.csv')
cmp_summary = json.loads((PORTION_DIR / 'comparison_summary.json').read_text())
portion_report = json.loads((PORTION_DIR / 'llm_eval_summary_portion.json').read_text())
baseline_report = json.loads((BASELINE_DIR / 'cost_report.json').read_text())

print('baseline ingredients:', len(baseline))
print('portion ingredients:', len(portion))
print('comparison summary:')
display(pd.Series(cmp_summary))

## 3. Classification tables (v3 pipeline paths)

In [ ]:
print('=== amount_kind ===')
display(portion['amount_kind'].value_counts())

print('=== retrieval_tier ===')
display(portion['retrieval_tier'].value_counts())

print('=== grams_status ===')
display(portion['grams_status'].value_counts())

print('=== pipeline_path (top 15) ===')
display(portion['pipeline_path'].value_counts().head(15))

In [ ]:
pivot = pd.crosstab(portion['amount_kind'], portion['grams_status'], margins=True)
display(pivot)

tier_pivot = pd.crosstab(portion['retrieval_tier'], portion['grams_status'].fillna('missing'))
display(tier_pivot)

## 4. Problem-size analysis — does portion-aware matching matter?

In [ ]:
comparison['baseline_resolved'] = comparison['grams_baseline_recomputed'].notna()
comparison['portion_resolved'] = comparison['grams'].notna()
comparison['newly_resolved'] = (~comparison['baseline_resolved']) & comparison['portion_resolved']
comparison['lost_resolution'] = comparison['baseline_resolved'] & (~comparison['portion_resolved'])

vol_count = comparison['amount_kind'].isin(['volume', 'count'])
matter = comparison[vol_count & (comparison['newly_resolved'] | comparison['fdc_id_changed'])]

print(f"Volume/count lines: {vol_count.sum()}")
print(f"Newly resolved (baseline→portion): {comparison['newly_resolved'].sum()}")
print(f"Lost resolution: {comparison['lost_resolution'].sum()}")
print(f"Gram rate baseline: {comparison['baseline_resolved'].mean():.1%}")
print(f"Gram rate portion:  {comparison['portion_resolved'].mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
rates = pd.Series({
    'baseline': comparison['baseline_resolved'].mean(),
    'portion v3': comparison['portion_resolved'].mean(),
})
rates.plot(kind='bar', ax=axes[0], color=['gray', 'seagreen'])
axes[0].set_title('Overall gram-resolvable rate')
axes[0].set_ylabel('fraction resolved')

for kind, ax in zip(['volume', 'count', 'mass'], [axes[1]] + [None, None]):
    pass

by_kind = comparison.groupby('amount_kind')[['baseline_resolved', 'portion_resolved']].mean()
by_kind.columns = ['baseline', 'portion v3']
by_kind.plot(kind='bar', ax=axes[1])
axes[1].set_title('Gram-resolvable rate by amount_kind')
axes[1].set_ylabel('fraction resolved')
plt.tight_layout()
plt.show()

## 5. Side-by-side examples — changed picks & newly resolved

In [ ]:
show_cols = [
    'recipe_id', 'ingredient_idx', 'ingredient_baseline', 'amount_kind',
    'retrieval_tier', 'llm_fdc_id_baseline', 'llm_fdc_id_portion',
    'grams_baseline_recomputed', 'grams', 'grams_status_baseline_recomputed',
    'grams_status', 'pipeline_path', 'llm_rationale_portion',
]
show_cols = [c for c in show_cols if c in comparison.columns]

print('=== Newly resolved (sample) ===')
display(comparison.loc[comparison['newly_resolved'], show_cols].head(20))

print('=== Lost resolution (sample) ===')
display(comparison.loc[comparison['lost_resolution'], show_cols].head(10))

print('=== fdc_id changed + volume/count (sample) ===')
changed = comparison[comparison['fdc_id_changed'] & vol_count]
display(changed[show_cols].head(20))

In [ ]:
candidates = pd.read_csv(PORTION_DIR / 'ingredient_candidates_top10.csv')
sample_keys = comparison.loc[comparison['newly_resolved'], ['recipe_id', 'ingredient_idx']].head(3)
for _, key in sample_keys.iterrows():
    rid, iidx = int(key['recipe_id']), int(key['ingredient_idx'])
    ing = comparison[(comparison['recipe_id']==rid) & (comparison['ingredient_idx']==iidx)].iloc[0]
    print('='*80)
    print(f"{ing['ingredient_baseline']} | amount_kind={ing.get('amount_kind')} tier={ing.get('retrieval_tier')}")
    print(f"baseline fdc={ing['llm_fdc_id_baseline']} -> portion fdc={ing['llm_fdc_id_portion']}")
    print(f"grams: {ing['grams_baseline_recomputed']} -> {ing['grams']} ({ing['grams_status']})")
    sub = candidates[(candidates['recipe_id']==rid) & (candidates['ingredient_idx']==iidx)]
    if 'portion_flag' in sub.columns:
        display(sub[['rank','fdc_id','description','lexical_dequant','dequant_sem','portion_flag','is_llm_pick']])
    else:
        display(sub.head(10))

## 6. Summary metrics

In [ ]:
summary_rows = {
    'metric': [
        'ingredients', 'fdc_id_agreement', 'gram_resolvable_rate',
        'newly_resolved', 'lost_resolution', 'abstain_rate', 'cost_usd',
    ],
    'baseline': [
        len(baseline), None,
        cmp_summary.get('baseline_gram_resolvable_rate'),
        None, None,
        baseline_report.get('abstain_rate'),
        baseline_report.get('cost_total_usd'),
    ],
    'portion_v3': [
        len(portion),
        cmp_summary.get('fdc_id_agreement_rate'),
        cmp_summary.get('portion_gram_resolvable_rate'),
        cmp_summary.get('n_newly_resolved'),
        cmp_summary.get('n_lost_resolution'),
        portion_report.get('abstain_rate'),
        portion_report.get('cost_total_usd'),
    ],
}
display(pd.DataFrame(summary_rows).set_index('metric'))

print('\nRetrieval tier usage (v3):', portion_report.get('retrieval_tier_counts'))
print('Gram status (v3):', portion_report.get('grams_status_counts'))